# Liftlog — train `ExerciseClassifier.mlmodel` (Phase 9C/9E)

Runs **on your Mac**, never on the Watch. Input: CSVs exported from the iPhone (Settings → Apple Watch → Motion Detection → Export).
Output: `ExerciseClassifier.mlmodel`, which you copy into `targets/watch/` and rebuild the app. Inference then runs on the Watch, offline.

* Windows: 2.5 s (125 samples @ 50 Hz), 50% overlap, 12 motion channels as an `MLMultiArray [1, 12, 125]`.
* Personal data (Calibrate + "Wrong exercise" corrections) is weighted ×`PERSONAL_WEIGHT`, and optionally fine-tuned on at the end.
* Held-out sets are whole sessions (overlapping windows of one set never land in both train and test).

In [ ]:
# Config
EXPORTS = ['data/*.csv']          # put your exported CSVs in ml/data/
WRIST = 'left'                    # train for the wrist you wear the Watch on (None = use all data)
PERSONAL_WEIGHT = 3.0             # how much more a personal (calibration/correction) window counts
FINETUNE_EPOCHS = 8               # extra epochs on personal data only (0 = off)
EPOCHS = 40
SEED = 0
OUT = 'ExerciseClassifier.mlmodel'

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from motion_data import CHANNELS, WINDOW, PERSONAL_KINDS, load_sessions, summarize, build_dataset, split_by_session, windows

torch.manual_seed(SEED)
sessions, files = load_sessions(EXPORTS)
print(f'{len(files)} files')
if WRIST:
    sessions = [s for s in sessions if s['wrist'] == WRIST]
labels = summarize(sessions)
assert len(labels) >= 2, 'need at least 2 labels (include some "other" = rest/walking sets)'
if 'other' not in labels:
    print('WARNING: no "other" (Other / Rest) sets. The watch will then force every detection into an exercise.')

In [ ]:
train_sessions, test_sessions = split_by_session(sessions, seed=SEED)
Xtr, ytr, wtr, _ = build_dataset(train_sessions, labels, PERSONAL_WEIGHT)
Xte, yte, _, gte = build_dataset(test_sessions, labels, 1.0) if test_sessions else (Xtr[:0], ytr[:0], wtr[:0], np.array([]))
mean = Xtr.mean(axis=(0, 2)); std = Xtr.std(axis=(0, 2)) + 1e-6
print('train windows', Xtr.shape, 'test windows', Xte.shape)

In [ ]:
class Net(nn.Module):
    """Small 1D CNN. Channel normalization is baked in, so the watch feeds raw deviceMotion values."""
    def __init__(self, n_classes, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean, dtype=torch.float32).view(1, len(CHANNELS), 1))
        self.register_buffer('std', torch.tensor(std, dtype=torch.float32).view(1, len(CHANNELS), 1))
        self.c1 = nn.Conv1d(len(CHANNELS), 32, 5, padding=2); self.b1 = nn.BatchNorm1d(32)
        self.c2 = nn.Conv1d(32, 64, 5, padding=2); self.b2 = nn.BatchNorm1d(64)
        self.c3 = nn.Conv1d(64, 64, 3, padding=1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):  # logits
        x = (x - self.mean) / self.std
        x = F.max_pool1d(F.relu(self.b1(self.c1(x))), 2)
        x = F.max_pool1d(F.relu(self.b2(self.c2(x))), 2)
        x = F.relu(self.c3(x)).mean(dim=2)
        return self.fc(self.drop(x))

def fit(model, X, y, w, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    X, y, w = torch.tensor(X), torch.tensor(y), torch.tensor(w)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        total = 0.0
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            # Light augmentation: amplitude jitter (different weights/tempo) + sensor noise.
            xb = X[idx] * (1 + 0.1 * torch.randn(len(idx), 1, 1)) + 0.01 * torch.randn_like(X[idx])
            loss = (F.cross_entropy(model(xb), y[idx], reduction='none') * w[idx]).sum() / w[idx].sum()
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f'epoch {epoch:3}  loss {total / len(X):.3f}')

model = Net(len(labels), mean, std)
fit(model, Xtr, ytr, wtr, EPOCHS, 1e-3)

In [ ]:
# Bias toward YOU: a few low-learning-rate epochs on personal sets only (calibration + corrections).
personal = [s for s in train_sessions if s['kind'] in PERSONAL_KINDS and s['exercise_id'] in labels]
if FINETUNE_EPOCHS and personal:
    Xp, yp, wp, _ = build_dataset(personal, labels, 1.0)
    print(f'fine-tuning on {len(personal)} personal sets / {len(Xp)} windows')
    fit(model, Xp, yp, wp, FINETUNE_EPOCHS, 2e-4)
else:
    print('no personal data yet: run Settings → Motion Detection → Calibrate on the phone, export, retrain')

In [ ]:
# Evaluate like the watch does: average window probabilities over each held-out SET, then top-1.
model.eval()
with torch.no_grad():
    probs = F.softmax(model(torch.tensor(Xte)), dim=1).numpy() if len(Xte) else np.zeros((0, len(labels)))
print(f'window accuracy: {(probs.argmax(1) == yte).mean():.1%}' if len(probs) else 'no held-out sets (record 3+ sets per exercise)')
set_rows = []
for sid in np.unique(gte):
    m = gte == sid
    p = probs[m].mean(0)
    set_rows.append((labels[yte[m][0]], labels[p.argmax()], p.max()))
if set_rows:
    correct = sum(t == q for t, q, _ in set_rows)
    unsure = sum(c < 0.65 for _, _, c in set_rows)
    print(f'set accuracy: {correct}/{len(set_rows)} = {correct / len(set_rows):.1%}   "not sure" (<0.65): {unsure}')
    confusion = np.zeros((len(labels), len(labels)), dtype=int)
    for t, q, _ in set_rows: confusion[labels.index(t), labels.index(q)] += 1
    print('rows = true, cols = predicted'); print(labels); print(confusion)

In [ ]:
import coremltools as ct

class Export(nn.Module):
    """Softmax on top so Core ML's classifier outputs probabilities."""
    def __init__(self, net): super().__init__(); self.net = net
    def forward(self, x): return F.softmax(self.net(x), dim=1)

traced = torch.jit.trace(Export(model).eval(), torch.zeros(1, len(CHANNELS), WINDOW))
mlmodel = ct.convert(
    traced,
    inputs=[ct.TensorType(name='motion', shape=(1, len(CHANNELS), WINDOW))],
    classifier_config=ct.ClassifierConfig(labels),
    convert_to='neuralnetwork',   # -> .mlmodel. If a future coremltools drops this, use 'mlprogram' and save .mlpackage
)
mlmodel.short_description = 'Liftlog wrist exercise classifier: 2.5 s of 50 Hz deviceMotion -> exercise probabilities'
mlmodel.input_description['motion'] = '[1, 12, 125] channel-major: ' + ','.join(CHANNELS) + ' (yaw relative to window start)'
mlmodel.user_defined_metadata['labels'] = ','.join(labels)
mlmodel.user_defined_metadata['wrist'] = str(WRIST)
mlmodel.save(OUT)
print('saved', OUT, '->  cp', OUT, '../targets/watch/')

In [ ]:
# Optional sanity check (macOS only): Core ML vs PyTorch on one held-out window.
import platform
if platform.system() == 'Darwin' and len(Xte):
    out = mlmodel.predict({'motion': Xte[:1]})
    probs_key = next(k for k, v in out.items() if isinstance(v, dict))
    print('coreml:', out['classLabel'], max(out[probs_key].values()))
    print('torch :', labels[probs[0].argmax()], probs[0].max())